In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv
import pandas as pd
from matplotlib.lines import Line2D
from scipy.integrate import quad
from functools import reduce
import operator
Cww = 1
Cgg = 0
Cbb = -1
cff = 0
mz = 91.188
mw = 80.3692
cw = mw/mz
sw = np.sqrt(1 - cw**2) 
alpha = 1/128
alpha_s = 0.11 #mz
hbar_c = 1.973e-13  # GeV·cm
N_Z = 6e12

Caa = Cww + Cbb
Caz = cw**2 * Cww - sw**2 * Cbb
Czz = cw**4 * Cww + sw**4 * Cbb

In [ ]:
df1 = pd.read_csv('./total.csv')
df2 = pd.read_csv('./1e-4.csv')

df = pd.concat([df1, df2], axis=0)

In [ ]:
ma_grid = df['MQ'].values
Lambda_grid = df['gstar'].values
xs = df['xs'].values

In [ ]:
#######################
##光子衰变函数
#######################
def Gamma_a_to_gg(ma_grid, Lambda_grid, fermions):
    Ceff = C_gamma_gamma_eff(ma_grid, fermions)
    return 4 * np.pi * alpha**2 * ma_grid**3 * np.real(Ceff * np.conj(Ceff)) / Lambda_grid**2 
def C_gamma_gamma_eff(ma_grid, fermions):
    total = np.zeros_like(ma_grid, dtype=complex)
    for f in fermions:
        term = (
            f["Nc"] * f["Qf"]**2 * cff * B1_tau(f["mf"], ma_grid) / (16 * np.pi**2)
        )
        total += term
    return Caa + total + 2 * Cww * alpha / (np.pi * sw**2) * B2_tau(mw, ma_grid)
def c_qq(Qq, T3q, mq, ma_array):
    ln_mu2_mq2 = np.log(mz**2 / mq**2)
    ln_mu2_mZ2 = np.log(mz**2 / mz**2)
    ln_mu2_mW2 = np.log(mz**2 / mw**2)
    g_vals = g_tau(mq, ma_array)
    term1 = -12 * Qq**2 * alpha**2 * Caa * (ln_mu2_mq2 + g_vals)
    term2 = -12 * alpha**2 / (sw**2 * cw**2) * Caz * Qq * (T3q - 2 * Qq * sw**2) * (ln_mu2_mZ2 + 1.5)
    term3 = -12 * alpha**2 / (sw**4 * cw**4) * Czz * (Qq**2 * sw**4 - T3q * sw**2 * Qq + 1/8) * (ln_mu2_mZ2 + 0.5)
    term4 = -3 * alpha**2 /(sw**4) * (ln_mu2_mW2 + 0.5) * Cww
    return term1 + term2 + term3 + term4

def g_tau(ml, ma_array):
    g_vals = np.empty_like(ma_array, dtype=complex)
    small_mask = ml**2 < ma_array**2
    large_mask = ml**2 > ma_array**2
    # 小极限
    log_term = np.log(ma_array[small_mask]**2 / ml**2) - 1j * np.pi
    g_vals[small_mask] = -1/6 * log_term**2 + 2/3
    # 大极限
    g_vals[large_mask] = 7/3
    # 相等情况
    g_vals[~(small_mask | large_mask)] =  5 + 4/3*(-0.5 + 1j * np.pi/2)
    return g_vals

def build_sm_fermions(ma_grid):
    fermions = []
    sm_data = [
        # name, Qf, mf, T3, Nc
        ("u_L",  2/3, 2.2e-3,  +0.5, 3), ("u_R",  2/3, 2.2e-3,  0.0,  3),
        ("d_L", -1/3, 4.7e-3,  -0.5, 3), ("d_R", -1/3, 4.7e-3,  0.0,  3),
        ("c_L",  2/3, 1.28,    +0.5, 3), ("c_R",  2/3, 1.28,    0.0,  3),
        ("s_L", -1/3, 0.096,   -0.5, 3), ("s_R", -1/3, 0.096,   0.0,  3),
        ("t_L",  2/3, 173,     +0.5, 3), ("t_R",  2/3, 173,     0.0,  3),
        ("b_L", -1/3, 4.18,    -0.5, 3), ("b_R", -1/3, 4.18,    0.0,  3),
        ("e_L", -1.0, 0.511e-3,-0.5, 1), ("e_R", -1.0, 0.511e-3, 0.0, 1),
        ("mu_L",-1.0, 0.1057,  -0.5, 1), ("mu_R",-1.0, 0.1057,   0.0, 1),
        ("tau_L",-1.0, 1.777,  -0.5, 1), ("tau_R",-1.0, 1.777,   0.0, 1),
    ]
    for name, Qf, mf, T3, Nc in sm_data:
        cff = c_qq(Qf, T3, mf, ma_array=ma_grid)  # <-- 支持2D
        fermions.append({"name": name, "Nc": Nc, "Qf": Qf, "mf": mf, "cff": cff})
    return fermions
def B1_tau(ml, ma_grid):
    tau = 4 * ml**2 / ma_grid**2
    return 1 - tau * f_tau(ml, ma_grid)**2

def B2_tau(ml, ma_grid):
    tau = 4 * ml**2 / ma_grid**2
    return 1 - (tau - 1) * f_tau(ml, ma_grid)**2

def f_tau(ml, ma_grid):
    tau = 4 * ml**2 / ma_grid**2
    result = np.empty_like(tau, dtype=complex)
    mask1 = tau >= 1
    mask2 = tau < 1
    result[mask1] = np.arcsin(1 / np.sqrt(tau[mask1]))
    result[mask2] = (np.pi/2 + 0.5j * np.log((1 + np.sqrt(1 - tau[mask2])) / (1 - np.sqrt(1 - tau[mask2]))))
    return result

In [ ]:
#######################
##轻子衰变函数
#######################
def Gamma_a_to_ll(ma_grid, ml, cell_eff, Lambda_grid):
    Gamma = np.zeros_like(ma_grid)
    mask = ma_grid >= 2 * ml
    m = ma_grid[mask]
    L = Lambda_grid[mask]
    kin = np.sqrt(1 - 4 * ml**2 / m**2)
    pref = m * ml**2 / (8 * np.pi * L**2)
    Gamma[mask] = pref * np.abs(cell_eff[mask])**2 * kin
    return Gamma

def compute_lepton_decay_widths(ma_grid, Lambda_grid):
    results = {}
    lepton_data = {
        "e":  (0.511e-3, -0.5),
        "mu": (0.1057,   -0.5),
        "tau":(1.777,    -0.5),
    }
    for name, (ml, T3) in lepton_data.items():
        c_eff = c_qq(Qq=-1.0, T3q=T3, mq=ml, ma_array=ma_grid)  # ma_array must be 2D
        Gamma = Gamma_a_to_ll(ma_grid, ml=ml, cell_eff=c_eff, Lambda_grid=Lambda_grid)
        results[name] = Gamma
    return results

In [ ]:
#######################
##轻夸克和重夸克衰变
#######################
def compute_light_quark_decay_widths(ma_grid, Lambda_grid):
    alpha_s_vals = alpha_s * np.ones_like(ma_grid)
    results = {
        "uu": np.zeros_like(ma_grid, dtype=complex),
        "dd": np.zeros_like(ma_grid, dtype=complex),
        "ss": np.zeros_like(ma_grid, dtype=complex)
    }
    quark_groups = {
        "uu": [("u_L", 2/3, 2.2e-3, +0.5), ("u_R", 2/3, 2.2e-3, 0.0)],
        "dd": [("d_L", -1/3, 4.7e-3, -0.5), ("d_R", -1/3, 4.7e-3, 0.0)],
        "ss": [("s_L", -1/3, 0.096, -0.5), ("s_R", -1/3, 0.096, 0.0)],
    }
    for label, states in quark_groups.items():
        mq = states[0][2]  # threshold
        sum_cqq = np.zeros_like(ma_grid, dtype=complex)
        for name, Qf, mf, T3 in states:
            sum_cqq += c_qq(Qf, T3, mf, ma_array=ma_grid)
        form_factor = 1 + (83 / 4) * alpha_s_vals / np.pi
        prefactor = 32 * np.pi * alpha_s_vals**2 * ma_grid**3 / Lambda_grid**2
        amplitude = Cgg + sum_cqq / (32 * np.pi**2)
        amplitude_sq = amplitude * np.conj(amplitude)
        
        mask = ma_grid >= 2 * mq
        result = np.zeros_like(ma_grid, dtype=complex)
        result[mask] = prefactor[mask] * form_factor[mask] * amplitude_sq[mask]
        results[label] = np.real(result)
    return results

def Gamma_a_to_QQ(ma_grid, mQ, cQQ_eff, Lambda_grid):
    Gamma = np.zeros_like(ma_grid)
    mask = ma_grid >= 2 * mQ
    m = ma_grid[mask]
    L = Lambda_grid[mask]
    kin = np.sqrt(1 - 4 * mQ**2 / m**2)
    pref = 3 * m * mQ**2 / (8 * np.pi * L**2)
    ce2 = np.abs(cQQ_eff[mask])**2
    Gamma[mask] = pref * ce2 * kin
    return Gamma

def compute_heavy_quark_decay_widths(ma_grid, Lambda_grid):
    results = {}
    quarks = {
        "cc": [("c_L", 2/3, 1.28, +0.5), ("c_R", 2/3, 1.28, 0.0)],
        "bb": [("b_L", -1/3, 4.18, -0.5), ("b_R", -1/3, 4.18, 0.0)],
    }
    for label, quark_states in quarks.items():
        Gamma_sum = np.zeros_like(ma_grid)
        for name, Qf, mf, T3 in quark_states:
            c_eff = c_qq(Qq=Qf, T3q=T3, mq=mf, ma_array=ma_grid)
            Gamma = Gamma_a_to_QQ(ma_grid, mQ=mf, cQQ_eff=c_eff, Lambda_grid=Lambda_grid)
            Gamma_sum += Gamma
        results[label] = Gamma_sum
    return results

In [ ]:
#######################
##pi介子衰变函数
#######################
def lambda_func(x, y, z):
    return x**2 + y**2 + z**2 - 2*(x*y + y*z + z*x)

def g_00(r):
    def integrand(z):
        return np.sqrt(1 - 4*r/z) * np.sqrt(lambda_func(1, z, r))
    z_min = 4 * r
    z_max = (1 - np.sqrt(r))**2
    if z_max <= z_min:
        return 0
    val, _ = quad(integrand, z_min, z_max)
    return 2 / (1 - r)**2 * val

def g_pm(r):
    def integrand(z):
        return np.sqrt(1 - 4*r/z) * (z - r)**2 * np.sqrt(lambda_func(1, z, r))
    z_min = 4 * r
    z_max = (1 - np.sqrt(r))**2
    if z_max <= z_min:
        return 0
    val, _ = quad(integrand, z_min, z_max)
    return 12 / (1 - r)**2 * val

def Gamma_a_to_3pi(ma_grid, Lambda_grid, f_pi, Cgg, cuu, cdd):
    m_pi = 0.135  # GeV
    mu, md = 2.2e-3, 4.7e-3  # u, d quark masses
    ma_flat = ma_grid.flatten()
    Lambda_flat = Lambda_grid.flatten()
    cuu_flat = cuu.flatten()
    cdd_flat = cdd.flatten()
    Gamma = np.zeros_like(ma_flat)

    for i, ma in enumerate(ma_flat):
        if ma < 3 * m_pi:
            continue
        r = (m_pi / ma)**2
        gab = g_00(r) + g_pm(r)
        delta_c = Cgg * (md - mu)/(md + mu) + (cuu_flat[i] - cdd_flat[i]) / (32 * np.pi**2)
        pref = (np.pi / 6) * ma * m_pi**4 / (Lambda_flat[i]**2 * f_pi**2)
        Gamma[i] = pref * delta_c**2 * gab

    return Gamma.reshape(ma_grid.shape)

In [ ]:
f_pi = 0.4/3.1415926  # GeV
cuu = np.real(c_qq(2/3, +0.5, 2.2e-3, ma_grid))
cdd = np.real(c_qq(-1/3, -0.5, 4.7e-3, ma_grid))
Gamma_3pi = Gamma_a_to_3pi(ma_grid, Lambda_grid, f_pi, Cgg, cuu, cdd)

In [ ]:
Gamma_gg = Gamma_a_to_gg(ma_grid, Lambda_grid, build_sm_fermions(ma_grid))
Gamma_ll = compute_lepton_decay_widths(ma_grid, Lambda_grid)
#Gamma_lqlq = compute_light_quark_decay_widths(ma_grid, Lambda_grid)
Gamma_qq = compute_heavy_quark_decay_widths(ma_grid, Lambda_grid)

In [ ]:
np.shape(Gamma_gg),np.shape(Gamma_ll['e']),np.shape(Gamma_qq['bb'])

In [ ]:
Gamma_ll_total   = reduce(operator.add, Gamma_ll.values())
#Gamma_lqlq_total = reduce(operator.add, Gamma_lqlq.values())
Gamma_qq_total   = reduce(operator.add, Gamma_qq.values())
Gamma_total  = Gamma_gg + Gamma_ll_total + Gamma_qq_total + Gamma_3pi #+ Gamma_lqlq_total
BR_3pi = Gamma_3pi / Gamma_total

In [ ]:
Ea = (mz**2 + ma_grid**2) / (2 * mz)
La = hbar_c * Ea / (ma_grid * Gamma_total)
# Decay probability functions
def P_decay(Lmin, Lmax, Ld):
    return np.exp(-Lmin / Ld) - np.exp(-Lmax / Ld)
ME = P_decay(450, 9999999e99999, La)
Prompt = P_decay(0, 1, La)

In [ ]:
# 创建一个DataFrame来保存所有衰变宽度
output_df = pd.DataFrame({
    'ma': ma_grid,
    'Lambda': Lambda_grid,
    'xs': xs,
    'xs_ME': xs*ME,
    'Gamma_gg': Gamma_gg,
    'Gamma_ee': Gamma_ll['e'],
    'Gamma_mumu': Gamma_ll['mu'],
    'Gamma_tautau': Gamma_ll['tau'],
    'Gamma_cc': Gamma_qq['cc'],
    'Gamma_bb': Gamma_qq['bb'],
    'Gamma_3pi': Gamma_3pi,
    'BR_3pi': BR_3pi,
    'BR_gg': Gamma_gg/Gamma_total,
    'BR_ee': Gamma_ll['e']/Gamma_total,
    'BR_mumu': Gamma_ll['mu']/Gamma_total,
    'BR_tautau': Gamma_ll['tau']/Gamma_total,
    'BR_cc': Gamma_qq['cc']/Gamma_total,
    'BR_bb': Gamma_qq['bb']/Gamma_total,
    'xs_gg': xs* Gamma_gg/Gamma_total,
    'xs_ee': xs* Gamma_ll['e']/Gamma_total,
    'xs_mumu': xs* Gamma_ll['mu']/Gamma_total,
    'xs_tautau': xs* Gamma_ll['tau']/Gamma_total,
    'xs_cc': xs* Gamma_qq['cc']/Gamma_total,
    'xs_bb': xs* Gamma_qq['bb']/Gamma_total,
    'xs_3pi': xs* BR_3pi,
    'prompt':Prompt,
    'xs_gg_prompt': xs * Prompt * Gamma_gg/Gamma_total,
    'xs_ee_prompt': xs * Prompt * Gamma_ll['e']/Gamma_total,
    'xs_mumu_prompt': xs * Prompt * Gamma_ll['mu']/Gamma_total,
    'xs_tautau_prompt': xs * Prompt * Gamma_ll['tau']/Gamma_total,
    'xs_cc_prompt': xs * Prompt * Gamma_qq['cc']/Gamma_total,
    'xs_bb_prompt': xs * Prompt * Gamma_qq['bb']/Gamma_total,
    'xs_3pi_prompt': xs* BR_3pi*Prompt,
})

# 写入CSV文件
output_df.to_csv('alp_decay_widths.csv', index=False)